# Full Pipeline Demo — market_tags, raw files to final content

**Uses the v3 tag definition** (race-relative expert rank + minimum coverage gate -- see `code/market_tags.py`'s module docstring or the main walkthrough notebook's Section 3b for why). Pool sizes below are 425 (Unlucky) / 534 (Overlooked), not the earlier 478/1,033.

This is a **시연 (live-run demonstration)**, not a conceptual explainer — every cell below actually executes on real data and writes a real intermediate file to `demo_pipeline_outputs/`, in the exact order data flows through production. Open each saved file as you go to show 조/최 concretely what exists at each hop.

**7 stages, in order:**

| Stage | What happens | Saved as |
|---|---|---|
| 0 | The 4 raw ingredients | `demo_pipeline_outputs/00_raw/` (copies of the real source files) |
| 1 | Odds → implied win probability | `01_odds_with_implied_win.csv` |
| 2 | Settled results → who actually placed | `02_outcomes_with_placed.csv` |
| 3 | Harville engine: win prob → implied PLACE probability | `03_implied_place.csv` |
| 4 | The big join (mirrors `02_build_unified.py`) | `04_unified_dataset_demo.csv` |
| 5 | Carve the two tag archetype pools | `05_unlucky_reference_pool_demo.csv`, `05_overlooked_reference_pool_demo.csv` |
| 6 | Score one live race through `market_tags.tag_race()` | (in-memory `TagResult` objects) |
| 7 | The actual content format that gets produced | `06_final_output_example.json` |

**One honesty note before starting:** stage 0's `fund_p` is *reused* from a prior scoring run, not re-derived from raw features in this demo — because the raw 10-variable feature file (`gate2_features_built.csv`) isn't available in this session's environment (see `PRODUCTION_PIPELINE.md` — in real production, `fund_p` comes live from `horse_rating.py`'s `score_race()`, bb_rating's own scoring step). Everything else below — the odds join, the placement logic, the Harville calc, the archetype filtering, the tagging — is derived from scratch, live, in this notebook.


In [ ]:
import sys, os, json
from pathlib import Path
import pandas as pd
import numpy as np

def find_project_root(marker="code/market_tags.py"):
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.append(cwd)
    candidates.append(cwd / "market_tags_walkthrough")
    candidates.extend(cwd.parents)
    for base in candidates:
        if (base / marker).exists():
            return base
    for known in (Path.home() / "Documents" / "market_tags_walkthrough",
                  Path.home() / "Desktop" / "market_tags_walkthrough"):
        if (known / marker).exists():
            return known.resolve()
    raise FileNotFoundError("Could not find market_tags_walkthrough -- see the main walkthrough notebook's setup cell for the full fallback chain.")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.append(str(PROJECT_ROOT / "code"))

DEMO_DIR = PROJECT_ROOT / "demo_pipeline_outputs"
(DEMO_DIR / "00_raw").mkdir(parents=True, exist_ok=True)

import market_tags as mt
from harville_engine import place_probs

pd.set_option("display.max_columns", 20)
print("Project root:", PROJECT_ROOT)
print("Demo outputs will be written to:", DEMO_DIR)


## Stage 0 — the 4 raw ingredients

| File | What it is | Where it comes from in production |
|---|---|---|
| `raw_win_odds.csv` | Pre-race win odds, one row per horse per race | RDS live odds snapshot (or the settled-odds public API for historical data, per `PRODUCTION_PIPELINE.md`) |
| `raw_exotic_outcomes.csv` | Settled results — winner/2nd/3rd per race | RDS race results |
| `raw_expert_score.csv` | Aggregated tipster picks (5/4/3/2/1 points, ranks 1-5, qualifying experts only) | RDS `expect_5horse` |
| `raw_fund_p_from_bb_rating.csv` | Each horse's win probability from the fundamental rating model | bb_rating's `horse_rating.py` → `score_race()` (reused here, not re-derived — see note above) |

We already know these first 3 files cover **8,660 of ~33,110 total races** — that's the real, documented scope limitation carried through every downstream number in this notebook.


In [ ]:
raw_odds = pd.read_csv("data/raw_win_odds.csv")
raw_outcomes = pd.read_csv("data/raw_exotic_outcomes.csv", dtype={"race_id": str})
raw_expert = pd.read_csv("data/raw_expert_score.csv", dtype={"race_id": str})
raw_fundp = pd.read_csv("data/raw_fund_p_from_bb_rating.csv", dtype={"race_id": str})

for name, df in [("raw_win_odds", raw_odds), ("raw_exotic_outcomes", raw_outcomes),
                  ("raw_expert_score", raw_expert), ("raw_fund_p_from_bb_rating", raw_fundp)]:
    df.to_csv(DEMO_DIR / "00_raw" / f"{name}.csv", index=False)
    print(f"{name}: {df.shape}")

print()
raw_odds.head(3)


## Stage 1 — odds → implied win probability

Standard pari-mutuel conversion: `implied_win ∝ 0.8 / odds` (the 0.8 accounts for the ~20% track takeout), then normalized so each race's implied win probabilities sum to 1. Also reconstructs `race_id` = date(8) + track_code(2) + race_no(2) — `01`=서울, `03`=부산경남, 제주 excluded (standing rule, confirmed via the track-code crosswalk).


In [ ]:
TRACK_MAP = {"서울": "01", "부산경남": "03"}
odds = raw_odds[raw_odds["track"] != "제주"].copy()
odds["track_code"] = odds["track"].map(TRACK_MAP)
odds["race_id"] = odds["date"].astype(str) + odds["track_code"] + odds["race_no"].astype(str).str.zfill(2)
odds = odds.rename(columns={"gate_no": "horse_num"})

odds["raw_prob"] = 0.8 / odds["odds"]
odds["implied_win"] = odds.groupby("race_id")["raw_prob"].transform(lambda x: x / x.sum())
odds = odds.drop(columns=["raw_prob"])

odds.to_csv(DEMO_DIR / "01_odds_with_implied_win.csv", index=False)
print("Shape:", odds.shape, "| unique races:", odds["race_id"].nunique())
odds.head(3)


## Stage 2 — settled results → who actually placed

연승식 (place-bet) rule: **top 2** finish for fields of 5-7 starters, **top 3** for fields of 8+, **no place bet offered** for fields of 4 or fewer (excluded). `starters` is a pipe-separated list — one row per race becomes one row per horse.


In [ ]:
exo = raw_outcomes.dropna(subset=["starters"]).copy()
exo["starters_list"] = exo["starters"].str.split("|")
starters_long = exo.explode("starters_list").rename(columns={"starters_list": "horse_num"})
starters_long["horse_num"] = starters_long["horse_num"].astype(int)

def placed_rule(row):
    n = row["n_starters"]
    h = row["horse_num"]
    if n <= 4:
        return np.nan
    elif n <= 7:
        return int(h in (row["winner"], row.get("second")))
    else:
        return int(h in (row["winner"], row.get("second"), row.get("third")))

starters_long["placed"] = starters_long.apply(placed_rule, axis=1)
outcomes = starters_long[["race_id", "horse_num", "n_starters", "placed", "race_date", "region"]].dropna(subset=["placed"])
outcomes.to_csv(DEMO_DIR / "02_outcomes_with_placed.csv", index=False)
print("Shape:", outcomes.shape, "| unique races:", outcomes["race_id"].nunique())
print("Overall placed rate:", round(outcomes["placed"].mean(), 4))
outcomes.head(3)


## Stage 3 — Harville engine: win prob → implied PLACE probability

This is the piece that makes the Unlucky/Overlooked *ratios* meaningful: it's not enough to know a horse's win odds, we need "given this market, how often should a horse THIS strong finish in the money?" `harville_engine.place_probs()` (lambda=0.8, the standard favorite-overestimate correction) answers exactly that, race by race.


In [ ]:
def implied_place_for_race(group, top):
    p = dict(zip(group["horse_num"], group["implied_win"]))
    return place_probs(p, top=top, lam=0.8)

n_by_race = outcomes.groupby("race_id")["n_starters"].first()

implied_rows = []
for race_id, group in odds.groupby("race_id"):
    if race_id not in n_by_race.index:
        continue
    n = n_by_race.loc[race_id]
    if n <= 4:
        continue
    top = 2 if n <= 7 else 3
    probs = implied_place_for_race(group, top)
    for horse_num, p in probs.items():
        implied_rows.append({"race_id": race_id, "horse_num": horse_num, "implied_place": p})

implied_place_df = pd.DataFrame(implied_rows)
implied_place_df.to_csv(DEMO_DIR / "03_implied_place.csv", index=False)
print("Shape:", implied_place_df.shape)
implied_place_df.head(3)


## Stage 4 — the big join (mirrors `02_build_unified.py`)

Join implied-win-odds + placement outcome + implied-place + expert score + fund_p, all on `race_id` + `horse_num`. **Watch the row/race count as each join happens — it should land on exactly the documented checkpoint (57,753 rows / 5,878 races) if this reconstruction is faithful to the real pipeline.**


In [ ]:
join1 = odds.merge(outcomes, on=["race_id", "horse_num"], how="inner")
print(f"odds + outcomes:        {join1.shape[0]:>7,} rows, {join1['race_id'].nunique():>5,} races")

join2 = join1.merge(implied_place_df, on=["race_id", "horse_num"], how="inner")
print(f"+ implied_place:        {join2.shape[0]:>7,} rows, {join2['race_id'].nunique():>5,} races")

join3 = join2.merge(raw_expert[["race_id", "horse_num", "expert_score"]], on=["race_id", "horse_num"], how="inner")
print(f"+ expert_score:         {join3.shape[0]:>7,} rows, {join3['race_id'].nunique():>5,} races")

unified_demo = join3.merge(raw_fundp[["race_id", "horse_num", "fund_p"]], on=["race_id", "horse_num"], how="inner")
print(f"+ fund_p:               {unified_demo.shape[0]:>7,} rows, {unified_demo['race_id'].nunique():>5,} races")

print(f"\nDocumented checkpoint:   57,753 rows, 5,878 races")
match = unified_demo.shape[0] == 57753 and unified_demo["race_id"].nunique() == 5878
print("MATCHES documented checkpoint exactly" if match else "does NOT match -- investigate before trusting downstream numbers")

unified_demo.to_csv(DEMO_DIR / "04_unified_dataset_demo.csv", index=False)
unified_demo.head(3)


## Stage 5 — carve the two tag archetype pools (v3 definition)

Compute each horse's expert_score against BOTH the population-absolute bar (`>=97` / `<=23`) AND its own race-relative third (`expert_score_race_rank`), plus a minimum-coverage gate -- the same v3 logic now shipped in `market_tags.py`. This is a tightened version of the original filter: an earlier version of this pool used only a population-wide tercile, and a real audit found that could let several horses in the *same* race get tagged at once (worst case: 5 Overlooked tags in one 14-horse race). See `code/market_tags.py`'s module docstring for the full diagnosis.


In [ ]:
d = unified_demo.copy()
d["fund_p_race_rank"] = d.groupby("race_id")["fund_p"].rank(pct=True)
d["log_odds"] = np.log(d["odds"])
# v3: expert_score judged both on the population-absolute bar AND relative to this
# race's own field, plus a minimum real-coverage gate -- mirrors market_tags.py exactly.
d["expert_score_race_rank"] = d.groupby("race_id")["expert_score"].rank(pct=True)
d["race_coverage"] = d.groupby("race_id")["expert_score"].transform(lambda s: (s > 0).sum())

eligible = d["race_coverage"] >= mt.MIN_EXPERT_COVERAGE
experts_like = (d["expert_score"] >= mt.EXPERT_HIGH_MIN) & (d["expert_score_race_rank"] >= mt.EXPERT_RACE_RANK_HIGH_MIN)
nobody_picked = (d["expert_score"] <= mt.EXPERT_LOW_MAX) & (d["expert_score_race_rank"] <= mt.EXPERT_RACE_RANK_LOW_MAX)
stats_worse = d["fund_p_race_rank"] <= mt.FUND_RANK_WORSE_MAX
stats_better = d["fund_p_race_rank"] >= mt.FUND_RANK_BETTER_MIN

unlucky_demo = d[eligible & experts_like & stats_worse].copy()
overlooked_demo = d[eligible & nobody_picked & stats_better].copy()

print(f"Unlucky pool:    n={len(unlucky_demo):,}  (documented checkpoint, v3 definition: n=425)")
print(f"Overlooked pool: n={len(overlooked_demo):,}  (documented checkpoint, v3 definition: n=534)")

# match market_tags.py's expected pool schema
keep_cols = ["race_id", "race_date", "region", "horse_num", "odds", "log_odds", "expert_score",
             "fund_p_race_rank", "placed"]
unlucky_demo_out = unlucky_demo[keep_cols].rename(columns={"odds": "odds_final"})
overlooked_demo_out = overlooked_demo[keep_cols].rename(columns={"odds": "odds_final"})

unlucky_demo_out.to_csv(DEMO_DIR / "05_unlucky_reference_pool_demo.csv", index=False)
overlooked_demo_out.to_csv(DEMO_DIR / "05_overlooked_reference_pool_demo.csv", index=False)


## Stage 6 — score one live race through `market_tags.tag_race()`

Point `market_tags.py` at the pools we just built (not the pre-built ones from the main walkthrough) and pull a real race to tag — this is the exact function call a daily production job would make.


In [ ]:
mt.TAG_CONFIG["unlucky"]["pool_path"] = str(DEMO_DIR / "05_unlucky_reference_pool_demo.csv")
mt.TAG_CONFIG["overlooked"]["pool_path"] = str(DEMO_DIR / "05_overlooked_reference_pool_demo.csv")

has_unlucky_race = d[eligible & experts_like & stats_worse]["race_id"]
demo_race = d[d["race_id"] == has_unlucky_race.iloc[0]].copy()
print(f"Scoring race {demo_race['race_id'].iloc[0]} live ({len(demo_race)} horses)\n")

horses = demo_race[["horse_num", "expert_score"]].copy()
horses["horse_num"] = horses["horse_num"].astype(int)
horses["odds_final"] = demo_race["odds"]
horses_list = horses.to_dict("records")
fund_p_list = demo_race["fund_p"].tolist()

results = mt.tag_race(horses_list, fund_p_list, k=10)
for r in results:
    if r.tag:
        print(r.explain())


**Small known cosmetic quirk, worth flagging honestly rather than hiding:** `horse_num` prints as `1.0` instead of `1` above. This is a pre-existing minor bug in `market_tags.py` itself — `DataFrame.iterrows()` upcasts a row to a single dtype when the row mixes an int column with float columns, so `horse_num` silently becomes a float inside `tag_race()`. It doesn't affect any number or decision logic, only the display text. Cheap fix if it's ever worth doing: cast `horse_num` to `int` explicitly in `TagResult.explain()`.

## Stage 7 — the actual content format that gets produced

This is the answer to "어떠한 형식의 콘텐츠 데이터가 생성되는지" — what a downstream product (app, web page, API) would actually receive for one tagged horse. Below: the raw Python object, then a clean JSON serialization ready to hand to a frontend or push to a database.


In [ ]:
tagged = [r for r in results if r.tag]
example = tagged[0] if tagged else results[0]

print("=== Raw TagResult object ===")
print(example)


In [ ]:
def tag_result_to_json(r):
    return {
        "horse_num": int(r.horse_num),
        "tag": r.tag,
        "tier": r.tier,
        "odds_final": float(r.odds_final),
        "expert_score": float(r.expert_score),
        "n_comps": int(r.n_comps),
        "comps_placed_rate": None if r.comps_placed_rate is None else round(float(r.comps_placed_rate), 4),
        "pool_placed_rate": None if r.pool_placed_rate is None else round(float(r.pool_placed_rate), 4),
        "customer_facing_text": r.explain(),
        "suppressed": r.n_comps < mt.MIN_COMPS if r.tag else None,
    }

output_json = [tag_result_to_json(r) for r in results if r.tag is not None]
with open(DEMO_DIR / "06_final_output_example.json", "w", encoding="utf-8") as f:
    json.dump(output_json, f, ensure_ascii=False, indent=2)

print("=== Final content format (06_final_output_example.json) ===")
print(json.dumps(output_json, ensure_ascii=False, indent=2))


## Recap — every stage, input → logic → output

| Stage | Input file(s) | Logic | Output file |
|---|---|---|---|
| 0 | (RDS / API pulls) | — | `00_raw/*.csv` |
| 1 | `raw_win_odds.csv` | `0.8/odds`, normalized per race | `01_odds_with_implied_win.csv` |
| 2 | `raw_exotic_outcomes.csv` | 연승식 top-2/top-3 rule | `02_outcomes_with_placed.csv` |
| 3 | stage 1 output | `harville_engine.place_probs(lam=0.8)` | `03_implied_place.csv` |
| 4 | stages 1-3 + expert + fund_p | inner join on race_id+horse_num | `04_unified_dataset_demo.csv` |
| 5 | stage 4 output | expert tercile × fund_p rank filter | `05_*_reference_pool_demo.csv` |
| 6 | stage 5 pools + one live race | `market_tags.tag_race()` | in-memory `TagResult` list |
| 7 | stage 6 results | `.explain()` + JSON serialization | `06_final_output_example.json` |

Every file above is sitting in `demo_pipeline_outputs/` next to this notebook — open any of them directly during a walkthrough to show exactly what exists at that point in the pipeline.
